# Limpieza y preparación de los datos

## 1. Importación de librerías y configuración de rutas

Importo las herramientas para leer los archivos JSON, manejar las rutas
y trabajar con tablas.

Localizo el archivo `.env` para identificar la raíz del proyecto y
establecer la ubicación de los datos originales. No necesito cargar
el token porque en este paso solo leeré archivos locales.

In [25]:
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

# Localizamos el archivo .env para identificar la raíz del proyecto
ruta_env = find_dotenv(usecwd=True)

if not ruta_env:
    raise FileNotFoundError("No se ha encontrado el archivo .env.")

# Construimos la ruta de los datos originales
raiz_proyecto = Path(ruta_env).resolve().parent
carpeta_raw = raiz_proyecto / "src" / "data" / "raw"

# Comprobamos que la carpeta existe
if not carpeta_raw.is_dir():
    raise FileNotFoundError(f"No existe la carpeta: {carpeta_raw}")

print("Carpeta de datos originales:", carpeta_raw.relative_to(raiz_proyecto))

Carpeta de datos originales: src\data\raw


In [5]:
# Localizamos los archivos descargados
archivos_json = sorted(carpeta_raw.glob("Pelis_*_pag_*.json"))

# Acumulamos las películas de todos los archivos
listas_peliculas = []

for ruta_archivo in archivos_json:
    with open(ruta_archivo, "r", encoding="utf-8") as f:
        datos = json.load(f)

    listas_peliculas.extend(datos["results"])

# Convertimos la lista en una tabla
df_peliculas = pd.DataFrame(listas_peliculas)

print("Archivos leídos:", len(archivos_json))
print("Filas del DataFrame:", len(df_peliculas))
display(df_peliculas.head())

Archivos leídos: 34243
Filas del DataFrame: 681868


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,NaN,"[35, 10749]",271517,Apartment Hunting,en,Apartment Hunting,,6.2092,/lEf4Vr8UB5XStOJR6yqewSIgk3b.jpg,2000-01-01,False,False,8.000,2
1,False,/wBQ2tSdXgAOAdH14VgXtN9qD0Ol.jpg,"[16, 10751, 10402]",49948,Fantasía 2000,en,Fantasia 2000,Sesenta años después de que Walt Disney creara...,6.1978,/2uTZzBQsmJqTY7JuHwTcYNN4WsX.jpg,2000-01-01,False,False,6.972,1406
2,False,/b7bQc3XDokQ5a1ZDna2i0sI2xTf.jpg,[53],122163,Jill Rips,en,Jill Rips,Un tipo duro se infiltra en una misión persona...,5.8018,/cfuc1GSUQaN56gc8y5nvkXwRkLF.jpg,2000-01-28,False,False,3.800,30
3,False,NaN,"[80, 28, 12]",453614,Codicia mortal,es,Codicia mortal,,5.7937,/udsl8xmwqjIQlgFUr8wAuSacpKO.jpg,2000-01-01,False,False,7.000,2
4,False,NaN,[],841159,Rata callejera,es,Rata callejera,,5.6884,/93RBEPgUjCzGkHSilNkF65UTzSE.jpg,2000-01-01,False,False,6.000,2


## 2. Comprobación de identificadores duplicados

Compruebo cuántas filas tienen un identificador de película repetido
y cuántos identificadores únicos contiene el DataFrame.

Todavía no elimino registros.

In [22]:
print("Filas totales:", len(df_peliculas))
print("IDs únicos:", df_peliculas["id"].nunique())
print("Filas con ID ausente:", df_peliculas["id"].isna().sum())
print("Repeticiones de ID:", df_peliculas["id"].duplicated().sum())

Filas totales: 681868
IDs únicos: 681868
Filas con ID ausente: 0
Repeticiones de ID: 0


El DataFrame contiene 681.868 registros, todos con un identificador
de película informado y único. No se detectan duplicados por `id`,
por lo que no es necesario eliminar filas por este motivo.

## 3. Comprobación de los meses y las páginas descargadas

Para cada mes del periodo 2000–2025, leo su primera página para
conocer el número de páginas esperado y compruebo que existen
los archivos correspondientes.

Esta revisión verifica los archivos de la extracción; no garantiza
que TMDB incluya toda la producción cinematográfica del periodo.

In [23]:
# Generamos los 312 meses del periodo de estudio
meses = pd.period_range("2000-01", "2025-12", freq="M")

meses_revisados = 0
incidencias = []

for mes in meses:
    fecha_inicio = mes.start_time.strftime("%Y-%m-%d")
    fecha_fin = mes.end_time.strftime("%Y-%m-%d")

    ruta_primera = carpeta_raw / (
        f"Pelis_{fecha_inicio}_{fecha_fin}_pag_1.json"
    )

    # Sin la primera página no podemos conocer el total esperado
    if not ruta_primera.exists():
        incidencias.append(f"{mes}: falta la primera página")
        continue

    with open(ruta_primera, "r", encoding="utf-8") as archivo:
        datos_mes = json.load(archivo)

    total_paginas = datos_mes["total_pages"]

    if total_paginas > 500:
        incidencias.append(f"{mes}: supera las 500 páginas")
        continue

    # Comprobamos que existe cada página esperada
    for pagina in range(1, total_paginas + 1):
        ruta_pagina = carpeta_raw / (
            f"Pelis_{fecha_inicio}_{fecha_fin}_pag_{pagina}.json"
        )

        if not ruta_pagina.exists():
            incidencias.append(f"{mes}: falta la página {pagina}")

    meses_revisados += 1

print("Meses esperados:", len(meses))
print("Meses revisados:", meses_revisados)
print("Incidencias:", len(incidencias))

# Mostramos como máximo las primeras diez incidencias
for incidencia in incidencias[:10]:
    print(incidencia)

Meses esperados: 312
Meses revisados: 312
Incidencias: 0


Se han revisado los 312 meses del periodo 2000–2025.
Todos disponen de su primera página y de los archivos correspondientes
a las páginas esperadas. No se han detectado archivos faltantes.

## 4. Revisión de la estructura del DataFrame

Examino las columnas, sus tipos de datos y el número de valores
no nulos para identificar posibles necesidades de limpieza.

In [7]:
df_peliculas.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 681868 entries, 0 to 681867
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   adult              681868 non-null  bool   
 1   backdrop_path      266022 non-null  str    
 2   genre_ids          681868 non-null  object 
 3   id                 681868 non-null  int64  
 4   title              681868 non-null  str    
 5   original_language  681868 non-null  str    
 6   original_title     681868 non-null  str    
 7   overview           681868 non-null  str    
 8   popularity         681868 non-null  float64
 9   poster_path        568030 non-null  str    
 10  release_date       681868 non-null  str    
 11  softcore           681868 non-null  bool   
 12  video              681868 non-null  bool   
 13  vote_average       681868 non-null  float64
 14  vote_count         681868 non-null  int64  
dtypes: bool(3), float64(2), int64(2), object(1), str(7)
memory usa

El DataFrame contiene 681.868 filas y 15 columnas. Se observan valores
nulos en las rutas de imágenes, cuya ausencia no justifica eliminar
películas para este análisis.

La fecha de estreno está almacenada como texto y deberá convertirse
a un tipo de fecha. También será necesario comprobar las cadenas
vacías y las listas de géneros vacías, que no se detectan como nulos
mediante `info()`.

## 5. Conversión de la fecha de estreno

Convierto la columna `release_date` de texto a fecha para facilitar
las comprobaciones del periodo y el análisis temporal.

In [8]:
# Convertimos la columna de texto a fecha
df_peliculas["release_date"] = pd.to_datetime(
    df_peliculas["release_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Comprobamos el resultado
print("Tipo de dato:", df_peliculas["release_date"].dtype)
print("Fechas ausentes o no válidas:", df_peliculas["release_date"].isna().sum())

Tipo de dato: datetime64[us]
Fechas ausentes o no válidas: 0


La columna `release_date` se ha convertido correctamente a un tipo
de fecha. No se han detectado fechas ausentes o no válidas.

In [24]:
print("Fecha mínima:", df_peliculas["release_date"].min())
print("Fecha máxima:", df_peliculas["release_date"].max())

Fecha mínima: 2000-01-01 00:00:00
Fecha máxima: 2025-12-31 00:00:00


La fecha mínima es el 1 de enero de 2000 y la máxima el 31 de
diciembre de 2025, por lo que todas las fechas de estreno están
dentro del periodo de estudio.

Muchas películas tienen el 1 de enero como fecha, y es probable que
en esos casos sea una fecha provisional y no la real. Por eso el
análisis temporal se hará por año de estreno y no por fecha exacta.

## 6. Comprobación de títulos vacíos

Cuento los títulos vacíos o formados únicamente por espacios.
Esta comprobación no modifica los datos.

In [10]:
(df_peliculas["title"].str.strip() == "").sum()

np.int64(0)

### Comprobación de otros campos de texto

Cuento los títulos originales, idiomas originales y sinopsis que
están vacíos o contienen únicamente espacios, sin modificar los datos.

In [11]:
print(
    "Títulos originales vacíos:",
    (df_peliculas["original_title"].str.strip() == "").sum()
)

print(
    "Idiomas originales vacíos:",
    (df_peliculas["original_language"].str.strip() == "").sum()
)

print(
    "Sinopsis vacías:",
    (df_peliculas["overview"].str.strip() == "").sum()
)

Títulos originales vacíos: 0
Idiomas originales vacíos: 0
Sinopsis vacías: 608265


No se detectan títulos originales ni idiomas originales vacíos.
Se encuentran 608.265 sinopsis vacías, aproximadamente el 89,2 %
del total.

Las consultas se realizaron en español, por lo que este resultado
no permite determinar si existe una sinopsis en otros idiomas.
Se conservan estas películas, ya que la sinopsis no es necesaria
para el análisis previsto.

## 7. Comprobación de géneros cinematográficos ausentes

Cuento las películas cuya lista `genre_ids` está vacía.
Una lista vacía indica que no hay géneros informados en los datos
descargados. Esta comprobación no modifica las películas.

In [12]:
sin_generos = (df_peliculas["genre_ids"].str.len() == 0)

print("Películas sin géneros informados:", sin_generos.sum())
print("Porcentaje:", round(sin_generos.mean() * 100, 2))

Películas sin géneros informados: 154099
Porcentaje: 22.6


Se detectan 154.099 películas con la lista `genre_ids` vacía,
equivalentes al 22,6 % del conjunto.

Se conservan estos registros para los análisis que no requieren
género cinematográfico. En las comparaciones por género se
documentará esta falta de información.

Las películas con `genre_ids` vacío se conservan y se consideran
sin género cinematográfico identificado. En las tablas y gráficos
se mostrarán con la etiqueta «Sin género».

## 8. Revisión de las variables numéricas

Examino las puntuaciones, el número de votos y la popularidad
mediante sus estadísticas descriptivas, sin modificar sus valores.

In [13]:
df_peliculas[["vote_average", "vote_count", "popularity"]].describe()

,vote_average,vote_count,popularity
count,681868.000000,681868.000000,681868.000000
mean,2.419224,31.289464,1.071949
std,3.285625,453.185970,4.093125
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.533900
50%,0.000000,0.000000,0.814900
75%,5.600000,2.000000,1.243300
max,10.000000,41744.000000,3247.003800


La distribución del número de votos está muy concentrada en valores
bajos: al menos la mitad de las películas no tiene votos y el 75 %
tiene como máximo dos.

Antes de decidir qué hacer con las puntuaciones, compruebo su
relación con la ausencia de votos.

Los valores elevados de votos y popularidad se conservan, ya que
ser extremos no demuestra que sean erróneos.

In [14]:
sin_votos = df_peliculas["vote_count"] == 0

print("Películas sin votos:", sin_votos.sum())
print("Porcentaje:", round(sin_votos.mean() * 100, 2))

Películas sin votos: 414488
Porcentaje: 60.79


### Relación entre puntuaciones y ausencia de votos

El 60,79 % de las películas no tiene votos. Se conservan para analizar
su distribución por géneros y años.

Compruebo si las películas sin votos tienen una puntuación de cero,
para evitar interpretar ese valor como una valoración negativa.

In [15]:
df_peliculas.loc[sin_votos, "vote_average"].value_counts(dropna=False)

vote_average
0.0     413975
6.0        101
8.0         86
7.0         84
10.0        79
5.0         50
4.0         29
9.0         20
1.0         19
3.0         16
2.0         15
7.5          4
4.5          3
5.5          2
6.5          2
9.5          1
8.5          1
3.5          1
Name: count, dtype: int64

### Puntuaciones de películas sin votos

Se detectan 513 películas con una puntuación distinta de cero pese
a tener cero votos registrados.

Se conservan los datos originales. Para analizar las valoraciones,
se utilizará una nueva columna que marque como ausente la puntuación
de cualquier película sin votos.

In [16]:
df_peliculas["valoracion"] = df_peliculas["vote_average"].mask(sin_votos)

print("Valoraciones ausentes:", df_peliculas["valoracion"].isna().sum())

Valoraciones ausentes: 414488


### Nota sobre la columna `valoracion`

La columna `valoracion` deja ausentes las puntuaciones de las películas
sin votos y las de cero con votos. El análisis final se centra en la
distribución por géneros y grupos de dirección, y no compara
valoraciones, por lo que esta columna no se utiliza más adelante.
Se conserva en el archivo preparado por si se retoma esa comparación.

El número de votos sí se usa después, como indicador de la visibilidad
de cada película.

### Puntuaciones de cero en películas con votos

Compruebo si existen películas que tengan votos registrados pero
una puntuación de cero. Si aparecen, revisaré esos casos antes
de decidir cómo tratarlos.

In [17]:
con_votos_puntuacion_cero = (
    (df_peliculas["vote_count"] > 0)
    & (df_peliculas["vote_average"] == 0)
)

print("Películas con votos y puntuación cero:",
      con_votos_puntuacion_cero.sum())

Películas con votos y puntuación cero: 929


Se detectan 929 películas con votos registrados y puntuación cero.
Reviso sus títulos y su número de votos antes de decidir cómo
tratar estas valoraciones.

In [18]:
df_peliculas.loc[
    con_votos_puntuacion_cero,
    ["id", "title", "vote_count", "vote_average"]
].sort_values("vote_count", ascending=False).head(10)

,id,title,vote_count,vote_average
180740,103133,BearCity 2: The Proposal,25,0.0
275777,376967,Home Invader,3,0.0
245401,255487,The Boyfriend,3,0.0
348406,472957,Madilim ang Gabi,3,0.0
291835,397117,MMXII,3,0.0
359036,559145,Fighting For A City,3,0.0
304203,115436,Hason Raja,2,0.0
241584,291399,The B League,2,0.0
254245,331827,Rouge Fougère,2,0.0
295398,375613,Wish You Were Here,2,0.0


### Tratamiento de las puntuaciones de cero con votos

Se detectan 929 películas con votos registrados y puntuación cero.
El máximo de votos entre estos casos es 25.

Como no se ha podido confirmar que esas puntuaciones representen
una valoración real, se consideran ausentes en la columna `valoracion`.
Se conservan las películas y los valores originales de `vote_average`.

In [19]:
df_peliculas["valoracion"] = df_peliculas["valoracion"].mask(
    con_votos_puntuacion_cero
)

print("Valoraciones ausentes:", df_peliculas["valoracion"].isna().sum())

Valoraciones ausentes: 415417


### Creación del año de estreno

Extraigo el año de la fecha de estreno para analizar la distribución
de las películas por géneros y grupos de dirección a lo largo del tiempo.
Se conserva también la fecha completa.

In [20]:
df_peliculas["anio_estreno"] = df_peliculas["release_date"].dt.year

display(df_peliculas[["title", "release_date", "anio_estreno"]].head())

,title,release_date,anio_estreno
0,Apartment Hunting,2000-01-01,2000
1,Fantasía 2000,2000-01-01,2000
2,Jill Rips,2000-01-28,2000
3,Codicia mortal,2000-01-01,2000
4,Rata callejera,2000-01-01,2000


### Guardado de los datos preparados

Guardo el DataFrame en formato Pickle, que conserva las fechas,
las listas de géneros y los tipos de las columnas.

Este archivo permite continuar el trabajo sin volver a leer todos
los JSON originales. Todavía falta incorporar la información
de las personas responsables de la dirección.

In [21]:
# Creamos la carpeta de datos procesados
carpeta_procesados = raiz_proyecto / "src" / "data" / "processed"
carpeta_procesados.mkdir(parents=True, exist_ok=True)

# Guardamos el DataFrame
ruta_peliculas = carpeta_procesados / "peliculas_preparadas.pkl"
df_peliculas.to_pickle(ruta_peliculas)

print("Archivo guardado en:", ruta_peliculas.relative_to(raiz_proyecto))

Archivo guardado en: src\data\processed\peliculas_preparadas.pkl
